In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import urllib.request

# Download small dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data = urllib.request.urlopen(url).read().decode('utf-8')

# Reduce size for speed
data = data[:30000]

# Create vocab
chars = sorted(list(set(data)))
char2idx = {u:i for i, u in enumerate(chars)}
idx2char = np.array(chars)

# Encode text
text_as_int = np.array([char2idx[c] for c in data])

# Create sequences
seq_length = 40
examples_per_epoch = len(text_as_int)//(seq_length+1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = sequences.map(split_input_target)

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

# Build model
vocab_size = len(chars)
embedding_dim = 128
rnn_units = 128

def build_model():
    model = tf.keras.Sequential([
        layers.Embedding(vocab_size, embedding_dim),
        layers.GRU(rnn_units, return_sequences=True),
        layers.Dense(vocab_size)
    ])
    return model

model = build_model()

# Loss
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss)

# Train
EPOCHS = 3
model.fit(dataset, epochs=EPOCHS)

# Text generation
def generate_text(model, start_string):
    num_generate = 200
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    temperature = 1.0

    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)

print(generate_text(model, "ROMEO: "))